# Tokenizer alignment experiments
Experiments to try to map token with one tokenizer to tokens with another tokenizer.

In [1]:
import sys
import os

# Get the current working directory
current_dir = os.getcwd()

# Get the parent directory
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))

# Add the parent directory to the system path
sys.path.append(parent_dir)
cwd = os.getcwd()
from models.tokenizer_aligner import TokenizerAligner
from transformers import AutoTokenizer
from datasets import load_dataset

/opt/homebrew/Caskroom/miniforge/base/envs/tokenizer_delete/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ModuleNotFoundError: No module named 'torch'

In [2]:
tokenizer_name = "t5-small"
tokenizer_t5 = AutoTokenizer.from_pretrained(
    tokenizer_name, cache_dir="./cache/models", model_max_length=2048
)

# tokenizer = T5Tokenizer.from_pretrained('t5-small')
tokenizer_name = "meta-llama/Meta-Llama-3-8B"
tokenizer_llama = AutoTokenizer.from_pretrained(tokenizer_name, model_max_length=2048)
tokenizer_llama.add_special_tokens({"pad_token": "[PAD]"})
dataset = "timdettmers/openassistant-guanaco"
data = load_dataset(dataset, split="train[:2%]")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Repo card metadata block was not found. Setting CardData to empty.


In [3]:
text = data[0]["text"]
text_tokenized_llama = tokenizer_llama(
    text, padding=True, truncation=True, add_special_tokens=True
)
text_tokenized_t5 = tokenizer_t5(
    text, padding=True, truncation=True, add_special_tokens=True
)
token_idx_mapped = TokenizerAligner().align_tokens(
    text, text_tokenized_llama, text_tokenized_t5
)
tokens_str_llama = tokenizer_llama.tokenize(
    text, padding=True, truncation=True, add_special_tokens=True
)
tokens_str_t5 = tokenizer_t5.tokenize(
    text, padding=True, truncation=True, add_special_tokens=True
)
token_idx_str_mapped = TokenizerAligner.map_tokens_to_str(
    token_idx_mapped, tokens_str_llama, tokens_str_t5
)

In [4]:
print(token_idx_mapped[-1])
print(token_idx_str_mapped)

([348], [367])
[(['###'], ['▁#', '##']), (['ĠHuman', ':'], ['▁Human', ':']), (['ĠCan'], ['▁Can']), (['Ġyou'], ['▁you']), (['Ġwrite'], ['▁write']), (['Ġa'], ['▁', 'a']), (['Ġshort'], ['▁short']), (['Ġintroduction'], ['▁introduction']), (['Ġabout'], ['▁about']), (['Ġthe'], ['▁the']), (['Ġrelevance'], ['▁relevance']), (['Ġof'], ['▁of']), (['Ġthe'], ['▁the']), (['Ġterm'], ['▁term']), (['Ġ"', 'mon', 'op', 'son', 'y', '"'], ['▁"', 'mon', 'ops', 'on', 'y', '"']), (['Ġin'], ['▁in']), (['Ġeconomics', '?'], ['▁economic', 's', '?']), (['ĠPlease'], ['▁Please']), (['Ġuse'], ['▁use']), (['Ġexamples'], ['▁examples']), (['Ġrelated'], ['▁related']), (['Ġto'], ['▁to']), (['Ġpotential'], ['▁potential']), (['Ġmon', 'op', 'son', 'ies'], ['▁mono', 'p', 'son', 'ies']), (['Ġin'], ['▁in']), (['Ġthe'], ['▁the']), (['Ġlabour'], ['▁labour']), (['Ġmarket'], ['▁market']), (['Ġand'], ['▁and']), (['Ġcite'], ['▁', 'cite']), (['Ġrelevant'], ['▁relevant']), (['Ġresearch', '.', '###'], ['▁research', '.', '#', '##']), (['

In [5]:
print(len(tokens_str_llama))
print(len(text_tokenized_llama["input_ids"]))
print(len(tokens_str_t5))
print(len(text_tokenized_t5["input_ids"]))

349
349
369
369


In [8]:
text_tokenized_llama = tokenizer_llama(
    text, padding=True, truncation=True, add_special_tokens=True
)
text_tokenized_t5 = tokenizer_t5(
    text, padding=True, truncation=True, add_special_tokens=True
)
tokens_str_llama = tokenizer_llama.tokenize(
    text, padding=True, truncation=True, add_special_tokens=True
)
tokens_str_t5 = tokenizer_t5.tokenize(
    text, padding=True, truncation=True, add_special_tokens=True
)
word_token_idx_llama = text_tokenized_llama.word_ids()
word_token_idx_t5 = text_tokenized_t5.word_ids()
words_llama = TokenizerAligner.text_to_words(
    text, text_tokenized_llama, word_token_idx_llama
)
words_t5 = TokenizerAligner.text_to_words(text, text_tokenized_t5, word_token_idx_t5)
words_idx_mapped, words_str_mapped = TokenizerAligner.map_words(words_llama, words_t5)
token_idx_mapped = TokenizerAligner.map_tokens(
    words_idx_mapped, word_token_idx_llama, word_token_idx_t5
)
token_idx_str_mapped = TokenizerAligner.map_tokens_to_str(
    token_idx_mapped, tokens_str_llama, tokens_str_t5
)

In [10]:
print(words_t5[:20])
print(words_llama[:20])

['###', 'human:', 'can', 'you', 'write', 'a', 'short', 'introduction', 'about', 'the', 'relevance', 'of', 'the', 'term', '"monopsony"', 'in', 'economics?', 'please', 'use', 'examples']
['###', 'human', ':', 'can', 'you', 'write', 'a', 'short', 'introduction', 'about', 'the', 'relevance', 'of', 'the', 'term', '"', 'monopsony', '"', 'in', 'economics']


In [12]:
token_idx_str_mapped

[(['###'], ['▁#', '##']),
 (['ĠHuman', ':'], ['▁Human', ':']),
 (['ĠCan'], ['▁Can']),
 (['Ġyou'], ['▁you']),
 (['Ġwrite'], ['▁write']),
 (['Ġa'], ['▁', 'a']),
 (['Ġshort'], ['▁short']),
 (['Ġintroduction'], ['▁introduction']),
 (['Ġabout'], ['▁about']),
 (['Ġthe'], ['▁the']),
 (['Ġrelevance'], ['▁relevance']),
 (['Ġof'], ['▁of']),
 (['Ġthe'], ['▁the']),
 (['Ġterm'], ['▁term']),
 (['Ġ"', 'mon', 'op', 'son', 'y', '"'], ['▁"', 'mon', 'ops', 'on', 'y', '"']),
 (['Ġin'], ['▁in']),
 (['Ġeconomics', '?'], ['▁economic', 's', '?']),
 (['ĠPlease'], ['▁Please']),
 (['Ġuse'], ['▁use']),
 (['Ġexamples'], ['▁examples']),
 (['Ġrelated'], ['▁related']),
 (['Ġto'], ['▁to']),
 (['Ġpotential'], ['▁potential']),
 (['Ġmon', 'op', 'son', 'ies'], ['▁mono', 'p', 'son', 'ies']),
 (['Ġin'], ['▁in']),
 (['Ġthe'], ['▁the']),
 (['Ġlabour'], ['▁labour']),
 (['Ġmarket'], ['▁market']),
 (['Ġand'], ['▁and']),
 (['Ġcite'], ['▁', 'cite']),
 (['Ġrelevant'], ['▁relevant']),
 (['Ġresearch', '.', '###'], ['▁research', '.', 

In [14]:
main_list = [
    "###",
    "human",
    ":",
    "can",
    "you",
    "give",
    "me",
    "an",
    "example",
    "of",
    "a",
    "python",
    "script",
    "that",
    "opens",
    "an",
    "api",
    "point",
    "and",
    "serves",
    "a",
    "string",
    "?###",
    "assistant",
    ":",
    "sure",
    "!",
    "here",
    "'s",
    "an",
    "example",
    "python",
    "script",
    "that",
    "uses",
    "the",
    "flask",
    "web",
    "framework",
    "to",
    "create",
    "a",
    "simple",
    "api",
    "endpoint",
    "that",
    "serves",
    "a",
    "string",
    ":",
    "```",
    "",
    "from",
    "flask",
    "import",
    "flask",
    "",
    "app",
    "=",
    "flask",
    "(__",
    "name",
    "__)",
    "@app",
    ".route",
    "('/')",
    "def",
    "hello",
    "_world",
    "():",
    "",
    "return",
    "'",
    "hello",
    ",",
    "world",
    "!'",
    "if",
    "__",
    "name",
    "__",
    "==",
    "'__",
    "main",
    "__':",
    "",
    "app",
    ".run",
    "()",
    "```",
    "",
    "in",
    "this",
    "script",
    ",",
    "we",
    "first",
    "import",
    "the",
    "flask",
    "class",
    "from",
    "the",
    "flask",
    "module",
    ".",
    "then",
    "we",
    "create",
    "a",
    "new",
    "instance",
    "of",
    "the",
    "flask",
    "class",
    ",",
    "using",
    "the",
    "__",
    "name",
    "__",
    "variable",
    "to",
    "specify",
    "the",
    "name",
    "of",
    "the",
    "application",
    ".",
    "\\",
    "next",
    ",",
    "we",
    "define",
    "a",
    "new",
    "route",
    "using",
    "the",
    "@",
    "app",
    ".route",
    "()",
    "decorator",
    ".",
    "this",
    "decorator",
    "tells",
    "flask",
    "to",
    "map",
    "requests",
    "to",
    "the",
    "root",
    "url",
    '("/")',
    "to",
    "the",
    "hello",
    "_world",
    "()",
    "function",
    ".",
    "\\",
    "finally",
    ",",
    "we",
    "use",
    "the",
    "if",
    "__",
    "name",
    "__",
    "==",
    "'__",
    "main",
    "__':",
    "block",
    "to",
    "start",
    "the",
    "flask",
    "application",
    "when",
    "the",
    "script",
    "is",
    "executed",
    ".",
    "by",
    "default",
    ",",
    "the",
    "application",
    "will",
    "run",
    "on",
    "port",
    "",
    "500",
    "0",
    ".",
    "\\",
    "you",
    "can",
    "run",
    "this",
    "script",
    "and",
    "test",
    "the",
    "api",
    "by",
    "opening",
    "a",
    "web",
    "browser",
    "and",
    "navigating",
    "to",
    "http",
    "://",
    "localhost",
    ":",
    "500",
    "0",
    "/.",
    "you",
    "should",
    "see",
    "a",
    "simple",
    '"',
    "hello",
    ",",
    "world",
    '!"',
    "message",
    "displayed",
    "in",
    "your",
    "browser",
    ".###",
    "human",
    ":",
    "what",
    "changes",
    "would",
    "you",
    "need",
    "to",
    "make",
    "to",
    "the",
    "code",
    "above",
    "to",
    "serve",
    "a",
    "json",
    "object",
    "instead",
    "of",
    "a",
    "string",
    "?###",
    "assistant",
    ":",
    "to",
    "serve",
    "a",
    "json",
    "object",
    "instead",
    "of",
    "a",
    "string",
    ",",
    "you",
    "can",
    "modify",
    "the",
    '"',
    "hello",
    "_world",
    '()"',
    "function",
    "to",
    "return",
    "a",
    "json",
    "response",
    "using",
    "the",
    "flask",
    '"',
    "jsonify",
    '"',
    "function",
    ".",
    "here",
    "'s",
    "an",
    "example",
    "of",
    "how",
    "to",
    "modify",
    "the",
    "previous",
    "code",
    "to",
    "serve",
    "a",
    "json",
    "object",
    ":",
    "```",
    "from",
    "flask",
    "import",
    "flask",
    ",",
    "jsonify",
    "",
    "app",
    "=",
    "flask",
    "(name",
    ")",
    "@app",
    ".route",
    "('/')",
    "def",
    "hello",
    "_world",
    "():",
    "data",
    "=",
    "{",
    "'m",
    "essage",
    "':",
    "'",
    "hello",
    ",",
    "world",
    "!'",
    "}",
    "return",
    "jsonify",
    "(data",
    ")",
    "if",
    "name",
    "==",
    "'m",
    "ain'",
    ":\na",
    "pp.",
    "run(",
    ")\n`",
    "``\n\ni",
    "n",
    "this",
    "updated",
    "version",
    "of",
    "the",
    "code,",
    "",
    "we",
    "import",
    "the",
    '"j',
    'sonify"',
    "",
    "function",
    "from",
    "flask",
    "and",
    "use",
    "it",
    "to",
    "create",
    "a",
    "json",
    "response.",
    "",
    "we",
    "define",
    "a",
    "dictionary",
    "called",
    '"d',
    'ata"',
    "",
    "that",
    "contains",
    "a",
    "single",
    "key-",
    "value",
    "pair,",
    "",
    "where",
    "the",
    "key",
    "is",
    '"m',
    'essage"',
    "",
    "and",
    "the",
    "value",
    "is",
    '"h',
    "ello,",
    "",
    "world!",
    '".\n\nt',
    "he",
    '"j',
    'sonify"',
    "",
    "function",
    "takes",
    "a",
    "dictionary",
    "as",
    "its",
    "argument",
    "and",
    "returns",
    "a",
    "json-",
    "formatted",
    "response",
    "with",
    "the",
    "appropriate",
    "content-",
    "type",
    "headers.",
    "",
    "when",
    "we",
    "visit",
    "the",
    "root",
    "url",
    "of",
    "our",
    "flask",
    "application,",
    "",
    "we",
    "will",
    "see",
    "a",
    "json",
    "object",
    "with",
    "the",
    '"m',
    'essage"',
    "",
    "key",
    "and",
    "its",
    "associated",
    "value.",
    "y",
    "ou",
    "can",
    "test",
    "this",
    "modified",
    "script",
    "in",
    "the",
    "same",
    "way",
    "as",
    "the",
    "previous",
    "example,",
    "",
    "by",
    "opening",
    "a",
    "web",
    "browser",
    "and",
    "navigating",
    "to",
    "http:",
    "//l",
    "ocalhost:",
    "5",
    "000",
    "/",
    ".",
    "this",
    "time,",
    "",
    "you",
    "should",
    "see",
    "a",
    "json",
    "object",
    "with",
    "the",
    '"m',
    'essage"',
    "",
    "key",
    "and",
    "its",
    "associated",
    "value",
    "displayed",
    "in",
    "your",
    "browser.",
    "",
]

In [13]:
compare_list = [
    "###",
    "human:",
    "can",
    "you",
    "give",
    "me",
    "an",
    "example",
    "of",
    "a",
    "python",
    "script",
    "that",
    "opens",
    "an",
    "api",
    "point",
    "and",
    "serves",
    "a",
    "string?###",
    "assistant:",
    "sure!",
    "here's",
    "an",
    "example",
    "python",
    "script",
    "that",
    "uses",
    "the",
    "flask",
    "web",
    "framework",
    "to",
    "create",
    "a",
    "simple",
    "api",
    "endpoint",
    "that",
    "serves",
    "a",
    "string:",
    "```",
    "from",
    "flask",
    "import",
    "flask",
    "app",
    "=",
    "flask(__name__)",
    "@app.route('/')",
    "def",
    "hello_world():",
    "return",
    "'hello,",
    "world!'",
    "if",
    "__name__",
    "==",
    "'__main__':",
    "app.run()",
    "```",
    "in",
    "this",
    "script,",
    "we",
    "first",
    "import",
    "the",
    "flask",
    "class",
    "from",
    "the",
    "flask",
    "module.",
    "then",
    "we",
    "create",
    "a",
    "new",
    "instance",
    "of",
    "the",
    "flask",
    "class,",
    "using",
    "the",
    "__name__",
    "variable",
    "to",
    "specify",
    "the",
    "name",
    "of",
    "the",
    "application.",
    "\\",
    "next,",
    "we",
    "define",
    "a",
    "new",
    "route",
    "using",
    "the",
    "@app.route()",
    "decorator.",
    "this",
    "decorator",
    "tells",
    "flask",
    "to",
    "map",
    "requests",
    "to",
    "the",
    "root",
    "url",
    '("/")',
    "to",
    "the",
    "hello_world()",
    "function.",
    "\\",
    "finally,",
    "we",
    "use",
    "the",
    "if",
    "__name__",
    "==",
    "'__main__':",
    "block",
    "to",
    "start",
    "the",
    "flask",
    "application",
    "when",
    "the",
    "script",
    "is",
    "executed.",
    "by",
    "default,",
    "the",
    "application",
    "will",
    "run",
    "on",
    "port",
    "5000.",
    "\\",
    "you",
    "can",
    "run",
    "this",
    "script",
    "and",
    "test",
    "the",
    "api",
    "by",
    "opening",
    "a",
    "web",
    "browser",
    "and",
    "navigating",
    "to",
    "http://localhost:5000/.",
    "you",
    "should",
    "see",
    "a",
    "simple",
    '"hello,',
    'world!"',
    "message",
    "displayed",
    "in",
    "your",
    "browser.###",
    "human:",
    "what",
    "changes",
    "would",
    "you",
    "need",
    "to",
    "make",
    "to",
    "the",
    "code",
    "above",
    "to",
    "serve",
    "a",
    "json",
    "object",
    "instead",
    "of",
    "a",
    "string?###",
    "assistant:",
    "to",
    "serve",
    "a",
    "json",
    "object",
    "instead",
    "of",
    "a",
    "string,",
    "you",
    "can",
    "modify",
    "the",
    '"hello_world()"',
    "function",
    "to",
    "return",
    "a",
    "json",
    "response",
    "using",
    "the",
    "flask",
    '"jsonify"',
    "function.",
    "here's",
    "an",
    "example",
    "of",
    "how",
    "to",
    "modify",
    "the",
    "previous",
    "code",
    "to",
    "serve",
    "a",
    "json",
    "object:",
    "```",
    "from",
    "flask",
    "import",
    "flask,",
    "jsonify",
    "app",
    "=",
    "flask(name)",
    "@app.route('/')",
    "def",
    "hello_world():",
    "data",
    "=",
    "{",
    "'message':",
    "'hello,",
    "world!'",
    "}",
    "return",
    "jsonify(data)",
    "if",
    "name",
    "=='main':",
    "app.run()",
    "```",
    "in",
    "this",
    "updated",
    "version",
    "of",
    "the",
    "code,",
    "we",
    "import",
    "the",
    '"jsonify"',
    "function",
    "from",
    "flask",
    "and",
    "use",
    "it",
    "to",
    "create",
    "a",
    "json",
    "response.",
    "we",
    "define",
    "a",
    "dictionary",
    "called",
    '"data"',
    "that",
    "contains",
    "a",
    "single",
    "key-value",
    "pair,",
    "where",
    "the",
    "key",
    "is",
    '"message"',
    "and",
    "the",
    "value",
    "is",
    '"hello,',
    'world!".',
    "the",
    '"jsonify"',
    "function",
    "takes",
    "a",
    "dictionary",
    "as",
    "its",
    "argument",
    "and",
    "returns",
    "a",
    "json-formatted",
    "response",
    "with",
    "the",
    "appropriate",
    "content-type",
    "headers.",
    "when",
    "we",
    "visit",
    "the",
    "root",
    "url",
    "of",
    "our",
    "flask",
    "application,",
    "we",
    "will",
    "see",
    "a",
    "json",
    "object",
    "with",
    "the",
    '"message"',
    "key",
    "and",
    "its",
    "associated",
    "value.",
    "you",
    "can",
    "test",
    "this",
    "modified",
    "script",
    "in",
    "the",
    "same",
    "way",
    "as",
    "the",
    "previous",
    "example,",
    "by",
    "opening",
    "a",
    "web",
    "browser",
    "and",
    "navigating",
    "to",
    "http://localhost:5000/.",
    "this",
    "time,",
    "you",
    "should",
    "see",
    "a",
    "json",
    "object",
    "with",
    "the",
    '"message"',
    "key",
    "and",
    "its",
    "associated",
    "value",
    "displayed",
    "in",
    "your",
    "browser.",
]

In [18]:
def map_words(main_list: list, compare_list: list):
    result_ids, result_words = [], []
    i = 0
    j = 0
    comb = [(i, j) for j in range(1, 11) for i in range(1, 11)]
    # we sorted the combinations to prioritize the most probables
    comb_sorted = sorted(comb, key=lambda x: x[0] + x[1], reverse=False)
    while i < len(main_list) and j < len(compare_list):
        asigned = 0
        for k, l in comb_sorted:
            if "".join(main_list[i : i + l]) == "".join(compare_list[j : j + k]):
                result_ids.append((list(range(i, i + l)), list(range(j, j + k))))
                result_words.append(
                    ("".join(main_list[i : i + l]), "".join(compare_list[j : j + k]))
                )
                i += l
                j += k
                asigned = 1
                break
        if asigned == 1:
            continue
        else:
            print(f"problema en i {i} y j{j}")
            print(main_list[i], "".join(compare_list[j]))
            i += 1
    return result_ids, result_words

In [19]:
result_ids, result_words = map_words(main_list, compare_list)

problema en i 358 y j270
== =='main':
problema en i 359 y j270
'm =='main':
problema en i 360 y j270
ain' =='main':
problema en i 361 y j270
:
a =='main':
problema en i 362 y j270
pp. =='main':
problema en i 363 y j270
run( =='main':
problema en i 364 y j270
)
` =='main':
problema en i 365 y j270
``

i =='main':
problema en i 366 y j270
n =='main':
problema en i 367 y j270
this =='main':
problema en i 368 y j270
updated =='main':
problema en i 369 y j270
version =='main':
problema en i 370 y j270
of =='main':
problema en i 371 y j270
the =='main':
problema en i 372 y j270
code, =='main':
problema en i 373 y j270
 =='main':
problema en i 374 y j270
we =='main':
problema en i 375 y j270
import =='main':
problema en i 376 y j270
the =='main':
problema en i 377 y j270
"j =='main':
problema en i 378 y j270
sonify" =='main':
problema en i 379 y j270
 =='main':
problema en i 380 y j270
function =='main':
problema en i 381 y j270
from =='main':
problema en i 382 y j270
flask =='main':
problema